# Credit Card Fraud Detection pomoću neuronskih mreža

**Tema:** Primena neuronskih mreža za predviđanje prevara u finansijama  
**Tip problema:** Binarna klasifikacija ekstremno nebalansiranih podataka  
**Okruženje:** Google Colab + TensorFlow/Keras

U ovom projektu se porede dva pristupa:
1. **Supervised MLP neuronska mreža** za klasifikaciju transakcija.
2. **Autoencoder** kao anomaly detection model koji uči normalne transakcije i prijavljuje neobične transakcije kao sumnjive.

Poenta projekta nije samo da se dobije visoka accuracy, jer kod ovog dataseta model koji sve proglasi regularnim već ima preko 99% tačnosti. Zato gledamo precision, recall, F1, PR-AUC, ROC-AUC i confusion matrix.


## 1. Instalacija i import biblioteka

Notebook je napravljen za Google Colab.  
Dataset se automatski preuzima preko TensorFlow-hosted kopije Kaggle/ULB skupa podataka.

Ako neka biblioteka već postoji u Colab-u, instalacija će se samo preskočiti ili brzo završiti.


In [ ]:
# Ako radiš u Colab-u, ova ćelija je dovoljna.
# Runtime -> Change runtime type -> GPU nije obavezno, ali može ubrzati trening.

import sys
!{sys.executable} -m pip install -q --upgrade scikit-learn joblib

import os
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    accuracy_score
)
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


## 2. Učitavanje podataka

Koristimo poznati **Credit Card Fraud Detection** dataset.  
U njemu su transakcije evropskih korisnika kartica iz septembra 2013. godine. Dataset ima:

- 284.807 transakcija,
- 492 prevarantske transakcije,
- 30 ulaznih atributa: `Time`, `Amount`, `V1`–`V28`,
- ciljnu promenljivu `Class`, gde je `1` prevara, a `0` regularna transakcija.

Kolone `V1`–`V28` su anonimizovane PCA transformacijom, pa ne znamo njihovo poslovno značenje. Zbog toga se projekat fokusira na modelovanje i evaluaciju, a ne na poslovno tumačenje svake pojedinačne PCA kolone.


In [ ]:
# TensorFlow-hosted kopija Kaggle/ULB dataseta.
# Prednost: radi direktno u Colab-u bez Kaggle API tokena.

zip_path = keras.utils.get_file(
    fname="creditcard.zip",
    origin="https://storage.googleapis.com/download.tensorflow.org/data/creditcard.zip",
    extract=True
)

csv_path = zip_path.replace(".zip", ".csv")
df = pd.read_csv(csv_path)

print("Putanja do CSV fajla:", csv_path)
print("Dimenzije dataseta:", df.shape)
df.head()


## 3. Analiza podataka

Najvažnija stvar kod ovog problema je **class imbalance**.  
Prevara ima veoma malo, pa accuracy nije dovoljna metrika.

Na primer, model koji kaže da nijedna transakcija nije prevara ima odličnu accuracy, ali je potpuno beskoristan jer ne detektuje nijednu prevaru.


In [ ]:
print("Kolone:")
print(df.columns.tolist())

print("\nNedostajuće vrednosti po kolonama:")
print(df.isna().sum().sort_values(ascending=False).head())

print("\nBroj duplikata:", df.duplicated().sum())

class_counts = df["Class"].value_counts().sort_index()
class_percent = df["Class"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "broj_transakcija": class_counts,
    "procenat": class_percent
})
summary.index = ["Regularna transakcija (0)", "Prevara (1)"]
summary


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Regularne", "Prevare"], class_counts.values)
plt.title("Raspodela klasa")
plt.ylabel("Broj transakcija")
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(df.loc[df["Class"] == 0, "Amount"], bins=80, alpha=0.8, label="Regularne")
plt.hist(df.loc[df["Class"] == 1, "Amount"], bins=80, alpha=0.8, label="Prevare")
plt.yscale("log")
plt.title("Distribucija iznosa transakcija - log skala")
plt.xlabel("Amount")
plt.ylabel("Broj transakcija")
plt.legend()
plt.show()


## 4. Preprocesiranje

Radimo sledeće:

1. Odvajamo `Class` kao ciljnu promenljivu.
2. Delimo podatke na train, validation i test skup.
3. Koristimo stratifikovanu podelu da se odnos regularnih/prevara očuva u svim skupovima.
4. Skaliramo ulazne podatke pomoću `RobustScaler`.
5. Skaler fitujemo samo na trening skupu da ne bi došlo do **data leakage**.

Test skup ostaje realno nebalansiran. To je važno jer u stvarnom sistemu prevare neće činiti 50% transakcija.


In [ ]:
features = [c for c in df.columns if c != "Class"]
target = "Class"

X = df[features].copy()
y = df[target].astype(int).copy()

# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "Fraud rate:", y_train.mean())
print("Validation:", X_val_scaled.shape, "Fraud rate:", y_val.mean())
print("Test:", X_test_scaled.shape, "Fraud rate:", y_test.mean())

n_features = X_train_scaled.shape[1]
n_features


## 5. Pomoćne funkcije za metrike i grafike

Za ovaj problem najbitnije metrike su:

- **Precision**: od svih transakcija koje je model označio kao prevaru, koliko je stvarno prevara.
- **Recall**: od svih stvarnih prevara, koliko ih je model pronašao.
- **F1-score**: balans između precision i recall.
- **PR-AUC**: korisnija metrika od ROC-AUC kod ekstremno nebalansiranih podataka.
- **Confusion matrix**: konkretno pokazuje FP i FN greške.

U praksi:
- **False Negative** je opasan jer prevara prođe neprimećeno.
- **False Positive** je neprijatan jer blokira legitimnog korisnika.


In [ ]:
def best_threshold_by_f1(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    # thresholds ima jedan element manje od precision/recall.
    precision_for_thr = precision[:-1]
    recall_for_thr = recall[:-1]

    f1_scores = 2 * precision_for_thr * recall_for_thr / (precision_for_thr + recall_for_thr + 1e-12)
    best_idx = int(np.nanargmax(f1_scores))

    return {
        "threshold": float(thresholds[best_idx]),
        "f1": float(f1_scores[best_idx]),
        "precision": float(precision_for_thr[best_idx]),
        "recall": float(recall_for_thr[best_idx])
    }


def evaluate_scores(model_name, y_true, scores, threshold, fn_cost=10, fp_cost=1):
    y_pred = (scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, scores)
    pr_auc = average_precision_score(y_true, scores)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    business_cost = fn * fn_cost + fp * fp_cost

    return {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "business_cost": int(business_cost)
    }


def plot_confusion(y_true, scores, threshold, title):
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks([0, 1], ["Regular", "Fraud"])
    plt.yticks([0, 1], ["Regular", "Fraud"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.show()


def plot_pr_curve(y_true, scores, title):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)

    plt.figure(figsize=(6, 4))
    plt.plot(recall, precision)
    plt.title(f"{title} - PR curve, AP={ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.grid(True)
    plt.show()


def plot_roc_curve(y_true, scores, title):
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)

    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"{title} - ROC curve, AUC={auc:.4f}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.grid(True)
    plt.show()


## 6. Baseline: model koji nikada ne prijavljuje prevaru

Ovaj baseline nam pokazuje zašto accuracy nije dovoljna.  
Ako model sve transakcije proglasi regularnim, imaće vrlo visoku accuracy, ali recall za prevaru biće 0.


In [ ]:
zero_scores = np.zeros(len(y_test))
baseline_result = evaluate_scores(
    "Baseline - sve regularno",
    y_test.values,
    zero_scores,
    threshold=0.5
)

pd.DataFrame([baseline_result])


## 7. Supervised MLP modeli

MLP je klasična feed-forward neuronska mreža za tabularne podatke.

Koristimo:
- `Dense` slojeve,
- `ReLU` aktivaciju,
- `BatchNormalization`,
- `Dropout`,
- `Adam` optimizator,
- `EarlyStopping` na osnovu validation PR-AUC.

Pored osnovnog MLP modela, testiramo i modele sa `class_weight`.  
`class_weight` daje veću težinu greškama nad fraud klasom, jer je ta klasa retka.


In [ ]:
def build_mlp(input_dim, hidden_units=(64, 32), dropout=0.2, learning_rate=1e-3):
    inputs = keras.Input(shape=(input_dim,), name="transaction_features")

    x = inputs
    for i, units in enumerate(hidden_units):
        x = layers.Dense(units, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"batch_norm_{i+1}")(x)
        x = layers.Dropout(dropout, name=f"dropout_{i+1}")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="fraud_probability")(x)

    model = keras.Model(inputs, outputs, name="FraudDetectionMLP")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(curve="PR", name="pr_auc"),
            keras.metrics.AUC(curve="ROC", name="roc_auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )
    return model


classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.values)
class_weight = {int(cls): float(w) for cls, w in zip(classes, weights)}

print("Class weights:", class_weight)


In [ ]:
EPOCHS = 40
BATCH_SIZE = 2048

configs = [
    {
        "name": "MLP_basic_no_class_weight",
        "hidden_units": (64, 32),
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "use_class_weight": False
    },
    {
        "name": "MLP_weighted_64_32",
        "hidden_units": (64, 32),
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "use_class_weight": True
    },
    {
        "name": "MLP_weighted_128_64_dropout",
        "hidden_units": (128, 64),
        "dropout": 0.3,
        "learning_rate": 5e-4,
        "use_class_weight": True
    }
]

trained_models = {}
histories = {}
val_results = []
test_results = [baseline_result]
thresholds = {}

for cfg in configs:
    print("\n" + "=" * 80)
    print("Trening:", cfg["name"])

    model = build_mlp(
        input_dim=n_features,
        hidden_units=cfg["hidden_units"],
        dropout=cfg["dropout"],
        learning_rate=cfg["learning_rate"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=5,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_train_scaled,
        y_train.values,
        validation_data=(X_val_scaled, y_val.values),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weight if cfg["use_class_weight"] else None,
        verbose=1
    )

    val_scores = model.predict(X_val_scaled, batch_size=4096).ravel()
    threshold_info = best_threshold_by_f1(y_val.values, val_scores)
    threshold = threshold_info["threshold"]

    test_scores = model.predict(X_test_scaled, batch_size=4096).ravel()

    val_row = evaluate_scores(
        cfg["name"] + " - validation",
        y_val.values,
        val_scores,
        threshold
    )
    test_row = evaluate_scores(
        cfg["name"] + " - test",
        y_test.values,
        test_scores,
        threshold
    )

    trained_models[cfg["name"]] = model
    histories[cfg["name"]] = history.history
    thresholds[cfg["name"]] = threshold
    val_results.append(val_row)
    test_results.append(test_row)

pd.DataFrame(test_results).sort_values("f1", ascending=False)


## 8. Krive učenja

Ako validation PR-AUC raste i zatim stagnira, EarlyStopping zaustavlja trening.  
Ako training metrika raste, a validation opada, to je znak overfitting-a.


In [ ]:
for name, history in histories.items():
    plt.figure(figsize=(7, 4))
    plt.plot(history.get("loss", []), label="train_loss")
    plt.plot(history.get("val_loss", []), label="val_loss")
    plt.title(f"Loss kroz epohe - {name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history.get("pr_auc", []), label="train_pr_auc")
    plt.plot(history.get("val_pr_auc", []), label="val_pr_auc")
    plt.title(f"PR-AUC kroz epohe - {name}")
    plt.xlabel("Epoch")
    plt.ylabel("PR-AUC")
    plt.legend()
    plt.grid(True)
    plt.show()


## 9. Autoencoder kao anomaly detection model

Autoencoder radi drugačije od MLP klasifikatora.

Ideja:
1. Treniramo ga samo na regularnim transakcijama.
2. On uči kako izgleda normalna transakcija.
3. Kada dobije neobičnu transakciju, rekonstrukciona greška je veća.
4. Ako je greška veća od izabranog threshold-a, transakcija se označava kao sumnjiva.

Ovo je dobar dodatak projektu jer pokazuje i polu-nadgledani/anomaly detection pristup, ne samo klasičnu klasifikaciju.


In [ ]:
def build_autoencoder(input_dim, encoding_dim=8, learning_rate=1e-3):
    inputs = keras.Input(shape=(input_dim,), name="transaction_features")

    x = layers.Dense(32, activation="relu")(inputs)
    x = layers.Dense(16, activation="relu")(x)
    encoded = layers.Dense(encoding_dim, activation="relu", name="encoded_space")(x)

    x = layers.Dense(16, activation="relu")(encoded)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(input_dim, activation="linear", name="reconstruction")(x)

    autoencoder = keras.Model(inputs, outputs, name="FraudAutoencoder")
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return autoencoder


normal_train = X_train_scaled[y_train.values == 0]
normal_val = X_val_scaled[y_val.values == 0]

autoencoder = build_autoencoder(n_features, encoding_dim=8, learning_rate=1e-3)

ae_history = autoencoder.fit(
    normal_train,
    normal_train,
    validation_data=(normal_val, normal_val),
    epochs=50,
    batch_size=2048,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=5,
            restore_best_weights=True
        )
    ],
    verbose=1
)

def reconstruction_error(model, X_data):
    reconstructed = model.predict(X_data, batch_size=4096)
    return np.mean(np.square(X_data - reconstructed), axis=1)

val_errors = reconstruction_error(autoencoder, X_val_scaled)
ae_threshold_info = best_threshold_by_f1(y_val.values, val_errors)
ae_threshold = ae_threshold_info["threshold"]

test_errors = reconstruction_error(autoencoder, X_test_scaled)
ae_test_result = evaluate_scores(
    "Autoencoder_anomaly_detection - test",
    y_test.values,
    test_errors,
    ae_threshold
)

test_results.append(ae_test_result)

print("Autoencoder threshold info:", ae_threshold_info)
pd.DataFrame([ae_test_result])


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ae_history.history["loss"], label="train_loss")
plt.plot(ae_history.history["val_loss"], label="val_loss")
plt.title("Autoencoder reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(test_errors[y_test.values == 0], bins=80, alpha=0.8, label="Regularne")
plt.hist(test_errors[y_test.values == 1], bins=80, alpha=0.8, label="Prevare")
plt.axvline(ae_threshold, linestyle="--", label="Threshold")
plt.yscale("log")
plt.title("Reconstruction error - Autoencoder")
plt.xlabel("MSE reconstruction error")
plt.ylabel("Broj transakcija")
plt.legend()
plt.show()


## 10. Finalna evaluacija modela

Threshold se bira na validation skupu, a finalna ocena se radi na test skupu.  
To je bitno jer test skup ne sme da utiče na izbor modela ili threshold-a.

U tabeli gledamo:
- F1 kao balans precision/recall,
- Recall ako nam je važnije da uhvatimo što više prevara,
- Precision ako nam je važnije da smanjimo lažne uzbune,
- Business cost kao pojednostavljenu poslovnu cenu grešaka.


In [ ]:
results_df = pd.DataFrame(test_results).drop_duplicates(subset=["model"], keep="last")
results_df = results_df.sort_values(["f1", "pr_auc"], ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Najbolji model biramo po validation F1 među supervised modelima.
val_df = pd.DataFrame(val_results)
best_validation_model_name = val_df.sort_values("f1", ascending=False).iloc[0]["model"].replace(" - validation", "")
print("Najbolji supervised model po validation F1:", best_validation_model_name)
print("Threshold:", thresholds[best_validation_model_name])

best_model = trained_models[best_validation_model_name]
best_threshold = thresholds[best_validation_model_name]
best_test_scores = best_model.predict(X_test_scaled, batch_size=4096).ravel()

plot_confusion(y_test.values, best_test_scores, best_threshold, f"Confusion matrix - {best_validation_model_name}")
plot_pr_curve(y_test.values, best_test_scores, best_validation_model_name)
plot_roc_curve(y_test.values, best_test_scores, best_validation_model_name)

print(classification_report(
    y_test.values,
    (best_test_scores >= best_threshold).astype(int),
    target_names=["Regularna", "Prevara"],
    zero_division=0
))


## 11. Threshold analiza

Klasifikaciona neuronska mreža vraća verovatnoću/rizik, a ne direktnu odluku.  
Threshold određuje od kog rizika kažemo: "ovo je prevara".

- Niži threshold: više transakcija se označava kao prevara → veći recall, više false positive grešaka.
- Viši threshold: model je stroži → manje false positive grešaka, ali može propustiti više prevara.

Zato threshold nije samo tehničko pitanje, nego poslovna odluka.


In [ ]:
val_scores_best = best_model.predict(X_val_scaled, batch_size=4096).ravel()

threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_rows = []

for thr in threshold_grid:
    row = evaluate_scores(
        f"threshold_{thr:.2f}",
        y_val.values,
        val_scores_best,
        thr
    )
    threshold_rows.append({
        "threshold": thr,
        "precision": row["precision"],
        "recall": row["recall"],
        "f1": row["f1"],
        "business_cost": row["business_cost"]
    })

threshold_df = pd.DataFrame(threshold_rows)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.axvline(best_threshold, linestyle="--", label="Izabrani threshold")
plt.title("Uticaj threshold-a na metrike")
plt.xlabel("Threshold")
plt.ylabel("Vrednost metrike")
plt.legend()
plt.grid(True)
plt.show()

threshold_df.sort_values("f1", ascending=False).head(10)


## 12. Analiza osetljivosti: permutation importance

Pošto su `V1`–`V28` PCA kolone, ne znamo njihovo originalno značenje.  
Ipak možemo proveriti koje kolone model najviše koristi.

Permutation importance radi ovako:
1. Izračuna se osnovni PR-AUC.
2. Jedna kolona se nasumično izmeša.
3. Ako metrika značajno padne, znači da je ta kolona važna za model.

Ovo nije savršeno objašnjenje neuronske mreže, ali je dovoljno intuitivno za seminarski projekat.


In [ ]:
# Da bi bilo brzo, koristi se subset validation skupa.
sample_size = min(15000, len(X_val_scaled))
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_val_scaled), size=sample_size, replace=False)

X_sample = X_val_scaled[sample_idx].copy()
y_sample = y_val.values[sample_idx].copy()

base_scores = best_model.predict(X_sample, batch_size=4096).ravel()
base_ap = average_precision_score(y_sample, base_scores)

importance_rows = []

for col_idx, col_name in enumerate(features):
    X_permuted = X_sample.copy()
    rng.shuffle(X_permuted[:, col_idx])

    permuted_scores = best_model.predict(X_permuted, batch_size=4096).ravel()
    permuted_ap = average_precision_score(y_sample, permuted_scores)

    importance_rows.append({
        "feature": col_name,
        "base_pr_auc": base_ap,
        "permuted_pr_auc": permuted_ap,
        "importance_drop": base_ap - permuted_ap
    })

importance_df = pd.DataFrame(importance_rows).sort_values("importance_drop", ascending=False)
importance_df.head(15)


In [ ]:
top_imp = importance_df.head(12).sort_values("importance_drop")

plt.figure(figsize=(8, 5))
plt.barh(top_imp["feature"], top_imp["importance_drop"])
plt.title("Top 12 najuticajnijih atributa - permutation importance")
plt.xlabel("Pad PR-AUC nakon mešanja kolone")
plt.ylabel("Atribut")
plt.grid(True)
plt.show()


## 13. Čuvanje rezultata i modela

Ova ćelija čuva:
- tabelu metrika,
- najbolji supervised model,
- autoencoder model,
- scaler.

U GitHub repo ne moraš nužno kačiti `.keras` fajlove ako su veliki, ali možeš ih sačuvati kao dokaz reprodukcije.


In [ ]:
outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

results_df.to_csv(outputs_dir / "model_comparison.csv", index=False)
importance_df.to_csv(outputs_dir / "permutation_importance.csv", index=False)

best_model.save(outputs_dir / "best_supervised_mlp.keras")
autoencoder.save(outputs_dir / "autoencoder.keras")
joblib.dump(scaler, outputs_dir / "robust_scaler.joblib")

print("Sačuvani fajlovi:")
for path in outputs_dir.iterdir():
    print("-", path)


## 14. Zaključak

U ovom projektu je prikazana detekcija prevara sa kreditnim karticama pomoću neuronskih mreža.

Najvažniji zaključci:

1. Problem je ekstremno nebalansiran, pa accuracy nije dovoljna metrika.
2. Supervised MLP može dobro da nauči obrasce prevare, ali threshold mora pažljivo da se bira.
3. `class_weight` pomaže modelu da više obraća pažnju na retku fraud klasu, ali može povećati broj lažnih uzbuna.
4. Autoencoder je koristan kao anomaly detection pristup jer uči normalno ponašanje i prijavljuje odstupanja.
5. Najbolji model se ne bira samo po accuracy, nego po odnosu precision/recall/F1 i po poslovnom kompromisu između false positive i false negative grešaka.

Za odbranu je najbitnije da znaš da objasniš:
- zašto je dataset problematičan,
- zašto accuracy nije dovoljna,
- šta rade MLP i Autoencoder,
- šta znači threshold,
- šta znače FP i FN u bankarskom sistemu.


## 15. Primer korišćenja modela nad jednom transakcijom

Ovo je samo demonstracija kako bi se model koristio nakon treninga.  
U realnom sistemu bi backend dobio podatke o transakciji, skalirao ih istim scaler-om i poslao modelu.


In [ ]:
def predict_fraud_for_row(raw_row, scaler, model, threshold, feature_names):
    row_df = pd.DataFrame([raw_row])[feature_names]
    row_scaled = scaler.transform(row_df)
    risk_score = float(model.predict(row_scaled, verbose=0).ravel()[0])
    predicted_class = int(risk_score >= threshold)

    return {
        "risk_score": risk_score,
        "threshold": threshold,
        "predicted_class": predicted_class,
        "label": "Prevara" if predicted_class == 1 else "Regularna transakcija"
    }

# Primer: uzimamo jednu transakciju iz test skupa.
example_row = X_test.iloc[0].to_dict()
predict_fraud_for_row(example_row, scaler, best_model, best_threshold, features)
